In [ ]:
# 识别手写数字：MNIST Softmax 多分类
from pathlib import Path
import gzip
import shutil

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


def resolve_project_root() -> Path:
    """在项目目录或仓库根目录启动 Notebook 都能定位数据。"""
    current = Path.cwd()
    if (current / "data" / "MNIST" / "raw").exists():
        return current
    candidate = current / "Chapter09_SoftmaxClassifier" / "MNISTHandwrittenDigitRecognition"
    if (candidate / "data" / "MNIST" / "raw").exists():
        return candidate
    raise FileNotFoundError("未找到 data/MNIST/raw；请从项目目录或仓库根目录启动 Notebook。")


def ensure_raw_mnist(data_root: Path) -> None:
    """将仓库内的官方 gzip 数据首次解压为 torchvision 可读取的 IDX 文件。"""
    raw = data_root / "MNIST" / "raw"
    for archive in raw.glob("*.gz"):
        destination = raw / archive.stem
        if not destination.exists():
            with gzip.open(archive, "rb") as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)


torch.manual_seed(42)
batch_size = 64
project_root = resolve_project_root()
data_root = project_root / "data"
ensure_raw_mnist(data_root)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(data_root, train=True, download=False, transform=transform)
test_dataset = datasets.MNIST(data_root, train=False, download=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = torch.nn.Linear(784, 512)
        self.l2 = torch.nn.Linear(512, 256)
        self.l3 = torch.nn.Linear(256, 128)
        self.l4 = torch.nn.Linear(128, 64)
        self.l5 = torch.nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        x = F.relu(self.l3(x))
        x = F.relu(self.l4(x))
        return self.l5(x)  # CrossEntropyLoss 内部完成 softmax


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)


def train() -> float:
    model.train()
    total_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def test() -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


train_losses, test_accuracies = [], []
for epoch in range(10):
    train_losses.append(train())
    test_accuracies.append(test())
    print(f"Epoch {epoch + 1:02d}/10 | loss={train_losses[-1]:.4f} | accuracy={test_accuracies[-1]:.2f}%")

images_dir = project_root / "images"
images_dir.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_losses, marker="o", label="Train loss")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Training loss")
axes[0].legend()
axes[1].plot(test_accuracies, marker="o", label="Test accuracy")
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="Test accuracy")
axes[1].legend()
fig.tight_layout()
fig.savefig(images_dir / "training_metrics.png", dpi=160)
plt.show()
